# Linguistic Forensic Audit: NER & POS Drift


This notebook measures how POS-tag distributions (style) and named entities (content) erode across paraphrasing iterations, directly informing **RQ1: Style vs. Content Decay**.

---

In [ ]:
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import (
    ALL_DATASETS, DATA_PARAPHRASED, PARAPHRASERS,
    SPACY_MODEL, FIGURES_DIR, EXPERIMENTS_DIR,
)

CACHE_DIR = PROJECT_ROOT / "data" / "processed"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw data     : {DATA_PARAPHRASED}")
print(f"Cache dir    : {CACHE_DIR}")
print(f"Datasets     : {ALL_DATASETS}")

---
## Task 1 — Data Processing & Version Alignment

**Goal:** Pair the original (T0) text of each article with its T1 paraphrased versions across multiple paraphrasers, using `(key, source, dataset)` as the composite identifier.

### 1a. Load all 7 datasets & concatenate

In [ ]:
dfs = []
for ds in ALL_DATASETS:
    fpath = DATA_PARAPHRASED / f"{ds}_paraphrased.csv"
    df = pd.read_csv(fpath)
    df["dataset"] = ds
    dfs.append(df)
    print(f"  {ds}: {len(df):>6,} rows, {df['version_name'].nunique()} versions")

corpus = pd.concat(dfs, ignore_index=True)

# Normalise the known typo: 'orignal' -> 'original'
typo_count = (corpus["version_name"] == "orignal").sum()
corpus["version_name"] = corpus["version_name"].replace("orignal", "original")
print(f"\nTotal corpus : {len(corpus):,} rows")
print(f"'orignal' typos normalised: {typo_count}")

### Version parser — robust, no naive underscore splitting

The regex `(.+?)(?:_\1)*` finds the shortest repeating base token.  
This correctly handles parenthesised tokens like `dipper(low)_dipper(low)` without splitting on internal structure.

In [ ]:
_BASE_RE = re.compile(r"(.+?)(?:_\1)*$")


def parse_version(version_str: str) -> tuple:
    """Return (base_token, repetition_count) for a version string."""
    if version_str in ("original", "orignal"):
        return ("original", 0)
    m = _BASE_RE.fullmatch(version_str)
    if not m:
        raise ValueError(f"Cannot parse version: '{version_str}'")
    base = m.group(1)
    count = (len(version_str) + 1) // (len(base) + 1)
    return (base, count)


# Verify against every unique version string in the corpus
print(f"{'version_name':<55} {'base_token':<20} {'tier'}")
print("-" * 80)
for v in sorted(corpus["version_name"].unique()):
    base, n = parse_version(v)
    tier = "T0" if base == "original" else f"T{n}"
    print(f"{v:<55} {base:<20} {tier}")

### 1b. Filter to target versions & pivot

We only need T0 (original) and the T1 pass from five paraphrasers:

| Output column | Version string | Paraphraser |
|---|---|---|
| `text_T0` | `original` | — |
| `text_chatgpt` | `chatgpt` | ChatGPT |
| `text_dipper_high` | `dipper(high)` | Dipper (High) |
| `text_dipper_low` | `dipper(low)` | Dipper (Low) |
| `text_pegasus_slight` | `pegasus(slight)` | Pegasus (Slight) |
| `text_pegasus_full` | `pegasus(full)` | Pegasus (Full) |

In [ ]:
VERSION_TO_COL = {
    "original":        "text_T0",
    "chatgpt":         "text_chatgpt",
    "dipper(high)":    "text_dipper_high",
    "dipper(low)":     "text_dipper_low",
    "pegasus(slight)": "text_pegasus_slight",
    "pegasus(full)":   "text_pegasus_full",
}

TEXT_COLS = list(VERSION_TO_COL.values())

# Filter to only the versions we need
subset = corpus[corpus["version_name"].isin(VERSION_TO_COL.keys())].copy()
subset["col_name"] = subset["version_name"].map(VERSION_TO_COL)
print(f"Rows after filtering to target versions: {len(subset):,}")
print(f"Version distribution:\n{subset['col_name'].value_counts().to_string()}")

In [ ]:
# Pivot: one row per (key, source, dataset) with 6 text columns
paired = subset.pivot_table(
    index=["key", "source", "dataset"],
    columns="col_name",
    values="text",
    aggfunc="first",
)

paired.columns.name = None
paired = paired.reset_index()

print(f"Shape before dropna: {paired.shape}")
print(f"NaN counts per column:\n{paired[TEXT_COLS].isna().sum().to_string()}")

# Drop rows missing any of the 4 assignment-required columns
required_cols = ["text_T0", "text_chatgpt", "text_dipper_high", "text_pegasus_slight"]
paired = paired.dropna(subset=required_cols)

print(f"\nShape after dropna on required cols: {paired.shape}")
print(f"Remaining NaNs (optional cols):\n{paired[TEXT_COLS].isna().sum().to_string()}")

### 1c. Deliverable

Print `paired.shape`, display 2 rows per dataset (14 total), and confirm the `dataset` column and `text_dipper_low` column are present.

In [ ]:
print(f"paired.shape = {paired.shape}")
print(f"\nColumns: {list(paired.columns)}")
print(f"\n'dataset' column present : {'dataset' in paired.columns}")
print(f"'text_dipper_low' present: {'text_dipper_low' in paired.columns}")

# 2 rows per dataset
sample = paired.groupby("dataset").head(2)
print(f"\nSample: 2 rows per dataset ({len(sample)} rows total)")

display_cols = ["key", "source", "dataset"] + TEXT_COLS
# Truncate text for readability
display_df = sample[display_cols].copy()
for col in TEXT_COLS:
    display_df[col] = display_df[col].apply(
        lambda x: x[:80] + "..." if isinstance(x, str) and len(x) > 80 else x
    )
display_df

In [ ]:
# Verify key consistency: each row's T0 and paraphrased texts come from the same article
print("Articles per dataset:")
print(paired.groupby("dataset").size().to_string())
print(f"\nTotal articles: {len(paired):,}")
print(f"Unique sources: {sorted(paired['source'].unique())}")

In [ ]:
# Cache the paired DataFrame
cache_path = CACHE_DIR / "paired_all.pkl"
paired.to_pickle(cache_path)
print(f"Cached to {cache_path} ({cache_path.stat().st_size / 1e6:.1f} MB)")

---

*To reload on subsequent runs:*
```python
paired = pd.read_pickle(CACHE_DIR / "paired_all.pkl")
```